## 415_check_WD_object_2.ipynb
* [#415](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/415)
* denna Notebook [415_check_WD_object_2](https://github.com/salgo60/Stockholm_Archipelago_Trail/tree/main/Notebook/415_check_WD_object_2.ipynb)

In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-07-02 22:39


In [31]:
# ==================================================
# SAT QA Dashboard
#
# Sprint 1
#
# One row = one SAT object
# ==================================================

from collections import defaultdict
import requests
import pandas as pd

URL = "https://map.stockholmarchipelagotrail.com/data/geojson/poi-concordance.json"

data = requests.get(URL).json()

#
# SAT objects (SSOT inside notebook)
#

sat_objects = defaultdict(lambda: {

    "wikidata": None,

    "osm": [],

    "qa": {},

    "status": "⚪",

    "action": "",

    "comment": ""

})

#
# Build object model
#

for external_id, sat_id in data["satIdOf"].items():

    if external_id.startswith("wikidata:"):

        sat_objects[sat_id]["wikidata"] = external_id.replace(
            "wikidata:",
            ""
        )

    elif external_id.startswith("osm:"):

        sat_objects[sat_id]["osm"].append(
            external_id
        )

print(f"Loaded {len(sat_objects)} SAT objects")

Loaded 590 SAT objects


In [32]:
print(data.keys())

dict_keys(['generated', 'license', 'satIdOf'])


In [33]:
#print(data["satIdOf"].keys())  
first = next(iter(data["satIdOf"].items()))
print(first)

('grillplatser:G-0254e7bf-dd61-49dc-81c4-784e9e7f1999', 'sat:poi:eb5j5')


In [34]:
#
# QA registry
#

QA_TESTS = []


def register(test):

    QA_TESTS.append(test)

    return test


def run_tests():

    for test in QA_TESTS:

        print(f"Running {test.__name__}")

        test()

In [35]:
@register
def qa001_wikidata_exists():

    for sat in sat_objects.values():

        sat["qa"]["QA001"] = "⚪"

    #
    # alive
    #

    for qid in alive:

        sat_id = wd_to_sat[qid]

        sat_objects[sat_id]["qa"]["QA001"] = "🟢"

    #
    # deleted
    #

    for qid in deleted:

        sat_id = wd_to_sat[qid]

        sat_objects[sat_id]["qa"]["QA001"] = "🔴"

        sat_objects[sat_id]["action"] = (
            "Create Wikidata item"
        )

        sat_objects[sat_id]["comment"] = (
            "Deleted Wikidata"
        )

In [36]:
@register
def qa002_redirect():

    for sat in sat_objects.values():

        sat["qa"]["QA002"] = "⚪"

    for item in redirects:

        sat_id = wd_to_sat[item["old"]]

        sat_objects[sat_id]["qa"]["QA002"] = "🟡"

        sat_objects[sat_id]["action"] = (
            "Update poi-concordance.json"
        )

        sat_objects[sat_id]["comment"] = (
            f'{item["old"]} → {item["new"]}'
        )

In [37]:
@register
def qa003_osm_exists():

    for sat in sat_objects.values():

        sat["qa"]["QA003"] = "⚪"

    for osm in osm_alive:

        sat_id = osm_to_sat[osm]

        sat_objects[sat_id]["qa"]["QA003"] = "🟢"

    for item in osm_deleted:

        sat_id = item["sat_id"]

        sat_objects[sat_id]["qa"]["QA003"] = "🔴"

        sat_objects[sat_id]["action"] = (
            "Fix OSM"
        )

        sat_objects[sat_id]["comment"] = (
            "Deleted OSM"
        )

In [38]:
@register
def qa004_backlink():

    for sat in sat_objects.values():

        sat["qa"]["QA004"] = "🟢"

    for item in osm_ref_mismatch:

        sat_id = item["sat_id"]

        sat_objects[sat_id]["qa"]["QA004"] = "🔴"

        sat_objects[sat_id]["action"] = (
            "Fix ref:stockholmarchipelagotrail"
        )

        sat_objects[sat_id]["comment"] = (
            f'Found {item["found"]}'
        )

In [39]:
run_tests()

Running qa001_wikidata_exists


NameError: name 'alive' is not defined

In [30]:
dashboard_df

,SAT,Wikidata,OSM,QA001,QA002,QA003,QA004,Action,Comment
0,sat:poi:aze59,None,1,⚪,⚪,⚪,⚪,,
1,sat:poi:55f8n,None,1,⚪,⚪,⚪,⚪,,
2,sat:poi:bcscm,Q28375395,1,⚪,⚪,⚪,⚪,,
3,sat:poi:9xkjj,Q28375391,1,⚪,⚪,⚪,⚪,,
4,sat:poi:c8z6n,None,1,⚪,⚪,⚪,⚪,,
...,...,...,...,...,...,...,...,...,...
585,sat:poi:cxh8j,Q136036748,0,⚪,⚪,⚪,⚪,,
586,sat:poi:z4mbp,Q136087206,0,⚪,⚪,⚪,⚪,,
587,sat:poi:fwaj8,Q136207068,0,⚪,⚪,⚪,⚪,,
588,sat:poi:awnk4,Q136207103,0,⚪,⚪,⚪,⚪,,


In [14]:
# ==================================================
# SAT QA Dashboard v0.1
# ==================================================

import pandas as pd

dashboard = {}

#
# Build dashboard
#

all_sat_ids = sorted(set(sat_to_wd) | set(sat_to_osm))

for sat_id in all_sat_ids:

    dashboard[sat_id] = {

        #
        # Identity
        #

        "SAT": sat_id,

        "Wikidata": sat_to_wd.get(sat_id),

        "OSM objects": len(sat_to_osm.get(sat_id, [])),

        #
        # QA
        #

        "QA001 WD exists": "⚪",

        "QA002 WD redirect": "⚪",

        "QA003 OSM exists": "⚪",

        "QA004 OSM backlink": "⚪",

        #
        # Result
        #

        "Status": "⚪",

        "Action": "",

        "Comment": ""

    }

#
# QA001
# Wikidata exists
#

for qid in alive:

    if qid in wd_to_sat:

        dashboard[
            wd_to_sat[qid]
        ]["QA001 WD exists"] = "🟢"

#
# QA002
# Redirect
#

for item in redirects:

    sat_id = wd_to_sat[item["old"]]

    dashboard[sat_id]["QA001 WD exists"] = "🟢"

    dashboard[sat_id]["QA002 WD redirect"] = "🟡"

    dashboard[sat_id]["Action"] = (
        "Update poi-concordance.json"
    )

    dashboard[sat_id]["Comment"] = (
        f'{item["old"]} → {item["new"]}'
    )

#
# Deleted Wikidata
#

for qid in deleted:

    if qid in wd_to_sat:

        sat_id = wd_to_sat[qid]

        dashboard[sat_id]["QA001 WD exists"] = "🔴"

        dashboard[sat_id]["Action"] = (
            "Create/replace Wikidata item"
        )

        dashboard[sat_id]["Comment"] = (
            "Deleted Wikidata item"
        )

#
# QA003
# OSM exists
#

for osm in osm_alive:

    if osm in osm_to_sat:

        dashboard[
            osm_to_sat[osm]
        ]["QA003 OSM exists"] = "🟢"

#
# Deleted OSM
#

for item in osm_deleted:

    sat_id = item["sat_id"]

    dashboard[sat_id]["QA003 OSM exists"] = "🔴"

    dashboard[sat_id]["Action"] = "Fix OSM"

    dashboard[sat_id]["Comment"] = (
        "Deleted OSM object"
    )

#
# QA004
# Backlink
#

for item in osm_ref_mismatch:

    sat_id = item["sat_id"]

    dashboard[sat_id]["QA004 OSM backlink"] = "🔴"

    dashboard[sat_id]["Action"] = (
        "Fix ref:stockholmarchipelagotrail"
    )

    dashboard[sat_id]["Comment"] = (
        f'Found: {item["found"]}'
    )

#
# Overall status
#

for row in dashboard.values():

    row["Status"] = "🟢"

    if (
        row["QA001 WD exists"] == "🔴"
        or row["QA003 OSM exists"] == "🔴"
        or row["QA004 OSM backlink"] == "🔴"
    ):

        row["Status"] = "🔴"

    elif row["QA002 WD redirect"] == "🟡":

        row["Status"] = "🟡"

#
# DataFrame
#

dashboard_df = pd.DataFrame(
    dashboard.values()
)

dashboard_df = dashboard_df.sort_values(
    [
        "Status",
        "SAT"
    ]
)

dashboard_df

AttributeError: 'list' object has no attribute 'get'

In [ ]:
*